# 📱 Aula 14 — População, amostra e parâmetro na base de usuários de um app

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de um aplicativo de música**. O app tem **5.000 usuários**, e o time quer saber **quanto tempo, em média, cada usuário passa no app por dia**. Com a Aula 14, essa pergunta ganha nomes precisos: os 5.000 usuários são a **população**, a média verdadeira é um **parâmetro** ($\mu$), e o que o time consegue medir é uma **amostra**, de onde sai uma **estimativa**.

Para poder conferir as respostas, este notebook faz algo que na vida real não dá para fazer: **cria a população inteira**. Assim conhecemos o parâmetro verdadeiro e podemos ver o quanto cada amostra erra.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas e criando a população

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**.

Depois cria o **sorteador** (`rng`) com a **semente 42** (que faz tudo sair sempre igual) e **inventa a população**:
- 5.000 usuários, dos quais **1.000 assinam o plano Premium** e **4.000 usam o plano Gratuito**;
- o tempo diário no app (em minutos) de cada um é sorteado de um modelo em que os assinantes Premium usam o app bem mais.

Por fim, calcula o **parâmetro** $\mu$: a média de **todos** os 5.000 usuários. Na vida real, ninguém conhece esse número — é justamente o que queremos estimar.

In [ ]:
import numpy as np                    # calculadora de números e sorteios
import matplotlib.pyplot as plt       # gráficos

rng = np.random.default_rng(42)       # sorteador, semente 42

N = 5000                                                       # tamanho da população
premium = np.zeros(N, dtype=bool)
premium[rng.permutation(N)[:1000]] = True                      # 1.000 usuários Premium, sorteados
uso = np.where(premium, rng.gamma(9, 10, N), rng.gamma(4, 5, N)).round(1)   # minutos por dia

mu = uso.mean()                                                # o PARÂMETRO (média da população)
print(f"População: {N} usuários ({premium.sum()} Premium, {(~premium).sum()} Gratuito)")
print(f"Parâmetro μ (média verdadeira): {mu:.2f} minutos por dia")
print(f"  média dos Premium:  {uso[premium].mean():.2f} min")
print(f"  média dos Gratuito: {uso[~premium].mean():.2f} min")

**Como ler:** o parâmetro é $\mu \approx 34$ minutos por dia. Os dois grupos são bem diferentes: Premium usa cerca de 89 minutos por dia, Gratuito cerca de 20. Guarde isso, porque vai importar na seção 3.

## 1. Parâmetro, estimativa e erro amostral

### A situação
O time não consegue medir os 5.000 usuários. Ele sorteia uma **amostra aleatória simples (AAS)** de **50 usuários**, sem reposição, e calcula a média deles.

### A pergunta
1. Quanto vale a estimativa $\bar{x}$? Quanto ela erra em relação a $\mu$?
2. Se o time sorteasse **outra** amostra, sairia o mesmo número?

### A ferramenta da aula
- **Parâmetro** ($\theta$): número que descreve a população — aqui, $\mu$;
- **Estimador** ($\hat\theta$): a regra que usamos na amostra — aqui, a média amostral $\bar{X}$;
- **Estimativa**: o valor do estimador numa amostra concreta;
- **Erro amostral**: $\epsilon = \hat\theta - \theta$, o erro que vem **só** do fato de olharmos uma parte da população.

### O que o código faz
Sorteia **5 amostras diferentes** de 50 usuários (`rng.choice(..., replace=False)` sorteia sem reposição) e, para cada uma, mostra a estimativa e o erro amostral.

In [ ]:
for i in range(1, 6):
    amostra = rng.choice(uso, size=50, replace=False)          # AAS de 50 usuários
    estimativa = amostra.mean()
    print(f"Amostra {i}: estimativa = {estimativa:6.2f} min   erro amostral = {estimativa - mu:+6.2f} min")

**Como ler:** cada amostra dá uma estimativa **diferente**, às vezes acima, às vezes abaixo de $\mu$. É isso que a aula quer dizer com "**o estimador é uma variável aleatória**": antes do sorteio, não sabemos qual valor ele vai dar. Por isso faz sentido perguntar pela **média** e pela **variância** do estimador — assunto da Aula 15.

## 2. Com reposição ou sem reposição?

### A situação e a pergunta
É o exemplo da aula: uma população com os números de **1 a 9** ($\mu = 5$), de onde tiramos amostras de **3** elementos. Com reposição, a amostra pode ser $(1, 1, 1)$ e errar 4 unidades; sem reposição, o pior caso é $(1, 2, 3)$, que erra 3. **Qual das duas formas de sortear erra menos, em geral?**

### O que o código faz
Sorteia **100.000 amostras** de cada jeito, calcula a média de cada uma e mede:
- o **maior erro** que apareceu;
- o **desvio padrão** das médias (quanto as estimativas "se espalham" em torno de $\mu = 5$).

In [ ]:
populacao = np.arange(1, 10)                                   # números de 1 a 9

com_rep = rng.choice(populacao, size=(100_000, 3), replace=True).mean(axis=1)
sem_rep = np.array([rng.choice(populacao, 3, replace=False).mean() for _ in range(100_000)])

for nome, medias in [("com reposição", com_rep), ("sem reposição", sem_rep)]:
    print(f"{nome}: maior erro = {np.abs(medias - 5).max():.1f}   desvio padrão das médias = {medias.std():.3f}")

**Como ler:** sem reposição, o maior erro foi 3 (contra 4 com reposição) e as médias se espalham **menos** em torno de 5. Faz sentido: sem reposição, cada elemento sorteado traz informação **nova** sobre a população. Quando a população é muito maior que a amostra (5.000 usuários e 50 sorteados), a diferença entre os dois jeitos quase desaparece.

## 3. Tipos de amostragem: AAS, estratificada e sistemática

### A situação
O time vai repetir a pesquisa com **100 usuários**. Há três formas de sortear:
- **AAS**: sorteia 100 usuários quaisquer;
- **Estratificada**: divide a população em **estratos** (Premium e Gratuito) e sorteia de cada um na proporção da população: **20** Premium (20% da base) e **80** Gratuito;
- **Sistemática**: com a lista de usuários em ordem, usa o fator $s = N/n = 5000/100 = 50$: sorteia um ponto de partida entre os 50 primeiros e pega um usuário a cada 50.

### A pergunta
**Qual dos três planos dá estimativas mais próximas de $\mu$?**

### O que o código faz
Repete cada plano **10.000 vezes** e mostra:
- a **média das estimativas** (se fica perto de $\mu$, o plano não "puxa" a estimativa para nenhum lado);
- o **desvio padrão das estimativas** (quanto menor, mais precisa é cada pesquisa);
- um gráfico com a distribuição das estimativas de cada plano.

In [ ]:
R, n = 10_000, 100
i_prem, i_grat = np.where(premium)[0], np.where(~premium)[0]   # posições de cada estrato
s = N // n                                                     # fator de sistematização = 50

aas = np.array([rng.choice(uso, n, replace=False).mean() for _ in range(R)])
estratificada = np.array([
    (uso[rng.choice(i_prem, 20, replace=False)].sum()          # 20 Premium
     + uso[rng.choice(i_grat, 80, replace=False)].sum()) / n   # 80 Gratuito
    for _ in range(R)])
sistematica = np.array([uso[rng.integers(0, s)::s].mean() for _ in range(R)])   # 1 a cada 50

print(f"Parâmetro μ = {mu:.2f}")
for nome, est in [("AAS", aas), ("Estratificada", estratificada), ("Sistemática", sistematica)]:
    print(f"{nome:14s} média das estimativas = {est.mean():.2f}   desvio padrão = {est.std():.2f}")

plt.hist(aas, bins=50, alpha=0.5, label="AAS", density=True)
plt.hist(estratificada, bins=50, alpha=0.5, label="Estratificada", density=True)
plt.axvline(mu, color="black", linestyle="--", label="μ verdadeiro")
plt.xlabel("Estimativa da média (minutos por dia)")
plt.ylabel("Densidade")
plt.title("10.000 pesquisas de 100 usuários cada")
plt.legend()
plt.show()

**Como ler:**
- Os três planos acertam **em média** (a média das estimativas fica em cerca de 34,0, colada em $\mu$): nenhum deles "puxa" a estimativa para um lado.
- A diferença está na **precisão**. A estratificada tem desvio padrão de cerca de **1,6 minuto**, a **metade** da AAS (cerca de 3,2). No gráfico, a distribuição da estratificada é bem mais estreita em torno de $\mu$.
- Por quê? Na AAS, o número de Premium na amostra varia de sorteio para sorteio (às vezes vêm 12, às vezes 28), e como os Premium usam o app muito mais, isso mexe muito na média. Na estratificada, vêm sempre exatamente 20 Premium e 80 Gratuito, na proporção certa.
- A sistemática ficou parecida com a AAS. Ela é útil por ser **mais simples de aplicar** (basta percorrer a lista), e funciona bem quando a ordem da lista não tem nenhum padrão ligado à variável.

**Lição:** quando existe um grupo que explica boa parte da variação (aqui, o tipo de plano), estratificar por ele dá estimativas mais precisas com o **mesmo** tamanho de amostra.

## 4. Viés de seleção: quando a amostra não representa a população

### A situação
Para economizar, alguém do time sugere: "em vez de sortear, vamos mandar a pesquisa por notificação dentro do app e usar quem responder". O problema: **quem usa mais o app vê mais notificações e responde mais**. No código, a chance de um usuário responder é proporcional ao tempo que ele passa no app.

### A pergunta
Essa pesquisa estima bem $\mu$? **E se aumentarmos a amostra**, o problema some?

### A ferramenta da aula
A aula chamou isso de **viés de seleção**: a forma de escolher a amostra favorece alguns elementos da população. Não é amostragem probabilística com chances iguais.

### O que o código faz
Simula **2.000 pesquisas por notificação**, com 100 e depois com 500 respostas, e compara a média das estimativas com $\mu$.

In [ ]:
chance_responder = np.minimum(1, uso / 100)                    # quem usa mais, responde mais

for n_resp in (100, 500):
    estimativas = []
    for _ in range(2000):
        responderam = uso[rng.random(N) < chance_responder]    # quem respondeu desta vez
        estimativas.append(rng.choice(responderam, n_resp, replace=False).mean())
    estimativas = np.array(estimativas)
    print(f"{n_resp} respostas: média das estimativas = {estimativas.mean():.2f}   "
          f"desvio padrão = {estimativas.std():.2f}   (μ verdadeiro = {mu:.2f})")

**Como ler:**
- A pesquisa por notificação estima cerca de **60 minutos por dia**, quase o **dobro** do verdadeiro $\mu \approx 34$. O erro não é azar: ele aparece em **todas** as repetições, sempre para cima, porque quem responde é quem mais usa o app.
- Aumentar de 100 para 500 respostas diminuiu o desvio padrão (de 3,9 para 1,7), mas a média das estimativas **continuou em 60**. A amostra maior só deixa a estimativa errada **mais precisa**.

**Lição:** viés de seleção não se corrige com mais dados. O que garante uma boa estimativa é **como** a amostra é escolhida (por sorteio), não só o tamanho dela.

## Para fechar

Neste notebook, o time de dados viu na prática os conceitos da Aula 14:

- **população** (os 5.000 usuários), **parâmetro** ($\mu$), **amostra**, **estimador** ($\bar X$), **estimativa** e **erro amostral**;
- o estimador é uma **variável aleatória**: cada amostra dá um valor diferente;
- **sem reposição** erra um pouco menos que com reposição;
- **estratificar** por um grupo que explica a variável (Premium × Gratuito) deixa as estimativas mais precisas;
- **viés de seleção** não se corrige aumentando a amostra.

Na **Aula 15**, olhamos para a **distribuição de todas as estimativas possíveis** — a **distribuição amostral** da média e da proporção.